In [1]:
import pandas as pd
import requests
import json

from google_play_scraper import Sort, reviews

In [19]:
result, continuation_token = reviews(
    'id.co.bri.brimo', # application unique tag in google playstore
    lang='id', # defaults to 'en'
    country='id', # defaults to 'us'
    sort=Sort.NEWEST, # defaults to Sort.NEWEST
    count=100, # defaults to 100
    filter_score_with=None # defaults to None(means all score)
)

df = pd.DataFrame(result)
df

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,537e91c1-c3fe-40f5-a57d-c174642ae371,Dandi Satiyawan,https://play-lh.googleusercontent.com/a-/ALV-U...,"Dengan adanya BRImo BRI, cukup mempermudah tra...",5,0,2.63.0,2024-07-16 13:51:59,Terima kasih atas ulasannya. Semoga aplikasi B...,2024-07-16 17:28:51,2.63.0
1,943c1c2b-4443-490d-a76a-07f8e94153bb,Yuni Astriani,https://play-lh.googleusercontent.com/a/ACg8oc...,Aplikasi nya bagus,5,0,2.63.0,2024-07-16 13:47:56,Terima kasih atas ulasannya. Semoga aplikasi B...,2024-07-16 17:29:03,2.63.0
2,515d25f9-d15a-4cf5-80d7-737fc44275f9,Risma Mariyam,https://play-lh.googleusercontent.com/a/ACg8oc...,bagus sangat membantu,5,0,2.63.0,2024-07-16 13:46:30,Terima kasih atas ulasannya. Semoga aplikasi B...,2024-07-16 17:29:15,2.63.0
3,733a4ad8-019f-4251-925f-4a405d826d50,Rudi Hartono,https://play-lh.googleusercontent.com/a/ACg8oc...,Gampang digunakan dan sangat membantu,5,0,2.63.0,2024-07-16 13:46:02,Terima kasih atas ulasannya. Semoga aplikasi B...,2024-07-16 17:29:30,2.63.0
4,92ae4cb9-ec87-4d03-ad7d-605f986970f8,Bayu gifcube,https://play-lh.googleusercontent.com/a-/ALV-U...,"Kenapa qris saya sering bermasalah, walaupun j...",3,0,2.63.0,2024-07-16 13:32:38,None,NaT,2.63.0
...,...,...,...,...,...,...,...,...,...,...,...
95,418e7c41-ff79-4d66-8726-383bbee5caca,Putra Buwono,https://play-lh.googleusercontent.com/a/ACg8oc...,Bagus,5,0,None,2024-07-16 08:37:16,"Hai Sobat BRI, terima kasih sudah menggunakan ...",2024-07-16 18:18:51,None
96,c0ad067c-3bbc-49e4-b201-9194dc3a5d9c,WAENI ANISA,https://play-lh.googleusercontent.com/a-/ALV-U...,Trimakasih Brimo,5,0,2.63.0,2024-07-16 08:37:16,"Hai Sobat BRI, terima kasih sudah menggunakan ...",2024-07-16 18:18:49,2.63.0
97,371af359-cef0-4028-b581-4d0de979cf78,Asilah Sholehah,https://play-lh.googleusercontent.com/a-/ALV-U...,Brimo ok,5,0,2.63.0,2024-07-16 08:34:47,"Hai Sobat BRI, terima kasih sudah menggunakan ...",2024-07-16 18:18:48,2.63.0
98,4eb8083c-0db2-49eb-8804-71a39978e902,panliraipal raipal,https://play-lh.googleusercontent.com/a/ACg8oc...,Membantu,5,0,2.63.0,2024-07-16 08:33:23,"Hai Sobat BRI, terima kasih sudah menggunakan ...",2024-07-16 18:18:47,2.63.0


In [ ]:
def review_analyze(text):
    payload = {
        "model" : "llama3",
        "messages" : [
            {"role" : "system",
             "content":f"""
                You are an expert in analyzing user reviews. Identify the key topics and sentiments of the following Playstore review for the BRImo mobile banking app by Bank Rakyat Indonesia, one of Indonesia's most reputable banks. Summarize the review and provide its sentiment.
                 """},
            {"role" : "user",
             "content": f"""
                Question: Identify the key topics of this playstore review for a mobile banking app of BRI (Bank Rakyat Indonesia) 
                          called BRImo which is one of the most reputable bank in Indonesia and what is the sentiments of these review?

                Review: "{text}"   

                Thought 1: Understand the content of the review and identify its key topics.
                Action 1: Analyze the review text to extract key topics.
                Observation 1: The key topics are...
                
                Thought 2: Analyze the tone and language of the review.
                Action 2: Determine the sentiment of the review text.
                Observation 2: The review text indicates positive/negative/neutral sentiment.

                Conclusion: Summarize the key topics and sentiment of the review using this format AND NOTHING ELSE BESIDES THIS: 
                - Topic (One of these value: 'User Experience', 'Network or Connection, 'Account/Login Issues, 'Banking System', 'Errors/Bugs'. 'Overall' if the review doesn't mention anything specifically)
                - Sentiment (Negative, Neutral, or Positive)
                - Explanation (1 to 2 sentence, explain briefly the analysis) 

                Return your answer in JSON format.
                """}
        ],
        "format": "json",  # constrain Ollama to emit valid JSON only, instead of hoping the prompt is enough
    }

    analysis_result = ""
    response = requests.post("http://localhost:11434/api/chat", json=payload)
    for message in response.iter_lines():
        jsonstr = json.loads(message)
        llm_answer = jsonstr["message"]["content"]
        analysis_result += llm_answer
    
    return analysis_result

    
results = []
for i, row in df.iterrows():
    review_id = row['reviewId']
    username = row['userName']
    datetime = row['at']
    rating = row['score']
    review_content = row['content']
    analysis = review_analyze(review_content)
    results.append({"review_id": review_id, "username": username, "datetime":datetime, "rating":rating, 
                    "review": review_content, "analysis": analysis})

results_df = pd.DataFrame(results)

In [21]:
results_df

,review_id,username,datetime,rating,review,analysis
0,537e91c1-c3fe-40f5-a57d-c174642ae371,Dandi Satiyawan,2024-07-16 13:51:59,5,"Dengan adanya BRImo BRI, cukup mempermudah tra...","{\n""Topic"": ""User Experience"",\n""Sentiment"": ""..."
1,943c1c2b-4443-490d-a76a-07f8e94153bb,Yuni Astriani,2024-07-16 13:47:56,5,Aplikasi nya bagus,"{\n""Topic"": ""User Experience"",\n""Sentiment"": ""..."
2,515d25f9-d15a-4cf5-80d7-737fc44275f9,Risma Mariyam,2024-07-16 13:46:30,5,bagus sangat membantu,"{\n""Topic"": ""User Experience"",\n""Sentiment"": ""..."
3,733a4ad8-019f-4251-925f-4a405d826d50,Rudi Hartono,2024-07-16 13:46:02,5,Gampang digunakan dan sangat membantu,"{\n""Topic"": ""User Experience"",\n""Sentiment"": ""..."
4,92ae4cb9-ec87-4d03-ad7d-605f986970f8,Bayu gifcube,2024-07-16 13:32:38,3,"Kenapa qris saya sering bermasalah, walaupun j...","{\n""Topic"": ""Network or Connection"",\n""Sentime..."
...,...,...,...,...,...,...
95,418e7c41-ff79-4d66-8726-383bbee5caca,Putra Buwono,2024-07-16 08:37:16,5,Bagus,"{\n""Topic"": ""Overall"",\n""Sentiment"": ""Positive..."
96,c0ad067c-3bbc-49e4-b201-9194dc3a5d9c,WAENI ANISA,2024-07-16 08:37:16,5,Trimakasih Brimo,"{\n""Topic"": ""Overall"",\n""Sentiment"": ""Positive..."
97,371af359-cef0-4028-b581-4d0de979cf78,Asilah Sholehah,2024-07-16 08:34:47,5,Brimo ok,"{\n""Topic"": ""Overall"",\n""Sentiment"": ""Neutral""..."
98,4eb8083c-0db2-49eb-8804-71a39978e902,panliraipal raipal,2024-07-16 08:33:23,5,Membantu,"{\n""Topic"": ""Overall"",\n""Sentiment"": ""Positive..."


In [ ]:
# Parsing logic lives in brimo_sentiment/parsing.py (unit tested — handles
# markdown-fenced and prose-wrapped JSON, which llama3 returns often enough
# in practice to be worth handling rather than only the happy path)
from brimo_sentiment.parsing import parse_llm_json

# Apply the function to parse the column
parsed_column = results_df["analysis"].apply(parse_llm_json)

# Convert the parsed column to a DataFrame
parsed_df = pd.json_normalize(parsed_column)

# Concatenate the new DataFrame with the original DataFrame
results_df = pd.concat([results_df, parsed_df], axis=1)
results_df

In [23]:
results_df.drop(columns='analysis')

,review_id,username,datetime,rating,review,Topic,Sentiment,Explanation,Topics
0,537e91c1-c3fe-40f5-a57d-c174642ae371,Dandi Satiyawan,2024-07-16 13:51:59,5,"Dengan adanya BRImo BRI, cukup mempermudah tra...",User Experience,Positive,"The review mentions that with BRImo BRI, trans...",NaN
1,943c1c2b-4443-490d-a76a-07f8e94153bb,Yuni Astriani,2024-07-16 13:47:56,5,Aplikasi nya bagus,User Experience,Positive,The review uses the phrase 'Aplikasi nya bagus...,NaN
2,515d25f9-d15a-4cf5-80d7-737fc44275f9,Risma Mariyam,2024-07-16 13:46:30,5,bagus sangat membantu,User Experience,Positive,The review uses the phrase 'bagus sangat memba...,NaN
3,733a4ad8-019f-4251-925f-4a405d826d50,Rudi Hartono,2024-07-16 13:46:02,5,Gampang digunakan dan sangat membantu,User Experience,Positive,The reviewer found the app easy to use and ver...,NaN
4,92ae4cb9-ec87-4d03-ad7d-605f986970f8,Bayu gifcube,2024-07-16 13:32:38,3,"Kenapa qris saya sering bermasalah, walaupun j...",Network or Connection,Negative,The reviewer mentions that their QRIS often ex...,NaN
...,...,...,...,...,...,...,...,...,...
95,418e7c41-ff79-4d66-8726-383bbee5caca,Putra Buwono,2024-07-16 08:37:16,5,Bagus,Overall,Positive,The reviewer's single-word review 'Bagus' indi...,NaN
96,c0ad067c-3bbc-49e4-b201-9194dc3a5d9c,WAENI ANISA,2024-07-16 08:37:16,5,Trimakasih Brimo,Overall,Positive,The review is a simple expression of gratitude...,NaN
97,371af359-cef0-4028-b581-4d0de979cf78,Asilah Sholehah,2024-07-16 08:34:47,5,Brimo ok,Overall,Neutral,The review is very brief and doesn't mention a...,NaN
98,4eb8083c-0db2-49eb-8804-71a39978e902,panliraipal raipal,2024-07-16 08:33:23,5,Membantu,Overall,Positive,The review contains a single word 'Membantu' w...,NaN


In [24]:
print(results_df.iloc[0,:]['Explanation'])

The review mentions that with BRImo BRI, transactions are made easier, indicating a positive sentiment towards the app's usability.


In [25]:
results_df.to_csv('results.csv', index=False)

Import from CSV

In [26]:
imported_df = pd.read_csv('results.csv')

In [27]:
list(imported_df['Topic'].unique())

['User Experience',
 'Network or Connection',
 'Overall',
 'Account/Login Issues',
 'Terpercaya',
 'Security',
 'Errors/Bugs',
 'Banking System',
 nan]